In [1]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from llmsource import llm
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

model = "qwen3.5:9b"
 

# # llm = ChatOllama(
# #     model=model,
# #     temperature=0
# # )
# class ChatState(TypedDict):
#     messages: Annotated[list[BaseMessage],add_messages]


# def chat_node(state:ChatState)->ChatState:
#     message = state['messages']
   
#     response = llm_with_tools.invoke(message)
    
#     return{
#         "messages":[response]
#     }
# checkpoint = MemorySaver()
# graph = StateGraph(ChatState)
# graph.add_node('chat_node',chat_node)

# # define edges
# graph.add_edge(START,'chat_node')
# graph.add_edge('chat_node',END)
# chatbot = graph.compile(checkpoint)

# # memory
# config = {
#     "configurable": {
#         "thread_id": "1"
#     }
# }



# init={"messages": [HumanMessage(content="My name is user 2")]}

# response = workflow.invoke(init,config)
# print(response['messages'][-1].content)        

In [2]:
import numexpr

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = numexpr.evaluate(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


In [3]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from llmsource import llm
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

model = "qwen3.5:9b"
tools = [calculator]
tools_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

# llm = ChatOllama(
#     model=model,
#     temperature=0
# )
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


def chat_node(state:ChatState)->ChatState:
    message = state['messages']
   
    response = llm_with_tools.invoke(message)
    
    return{
        "messages":[response]
    }
checkpoint = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_node('tools',tools_node)

graph.add_conditional_edges('chat_node',tools_condition)

# define edges
graph.add_edge(START,'chat_node')
graph.add_edge('tools','chat_node')

# graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpoint)

# memory
config = {
    "configurable": {
        "thread_id": "1"
    }
}




init={"messages": [HumanMessage(content="what is 34 * 6587878 ")]}

response = chatbot.invoke(init,config)

# print(response['messages'][-1].content)      
for message in response["messages"]:
    print(type(message).__name__)
    print("content:", message.content)

    if hasattr(message, "tool_calls"):
        print("tool_calls:", message.tool_calls)

    print("-" * 50)

HumanMessage
content: what is 34 * 6587878 
--------------------------------------------------
AIMessage
content: 
tool_calls: [{'name': 'calculator', 'args': {'expression': '34 * 6587878'}, 'id': '339e8229-1a3d-47bd-946e-6d84f7992b09', 'type': 'tool_call'}]
--------------------------------------------------
ToolMessage
content: 223987852
--------------------------------------------------
AIMessage
content: 34 multiplied by 6,587,878 is **223,987,852**.
tool_calls: []
--------------------------------------------------


The result of 34 multiplied by 6,587,878 is **223,987,852**.
